# 02 · Treinamento GRPO replicado

**GPU T4 ou superior.** Use o mesmo STUDY_ID dos notebooks anteriores.
Treina none/cot_monitor/summary_monitor em low/high, com 20 sementes:
120 treinos de 120 passos. O monitor de recompensa é distinto do monitor
independente. Reexecutar retoma checkpoints e mantém jobs concluídos.

**Diagnóstico DEV sem nova inferência:** se a triagem já falhou, o notebook mostra os dados salvos e não inicializa a GPU. O limite de 65% não é alterado.


In [ ]:
# Configuração operacional. Os tamanhos, sementes e critérios científicos são fixos.
from pathlib import Path
import hashlib, importlib.util, json, os, re, subprocess, sys
SOURCE_COMMIT = "fde856f895c841274b93b34f4675642508af102f"
SETUP_COMMIT = "4a2a133aff6638c910661b01821eeb43b72b4db4"
SCRIPT_SHA256 = "0ccc8cd66c429c116fe8a3ddeaee373d9dd72ec2e176d9dc10bc032da6c60f8b"
SUPPORT_SHA256 = "dacc3ff81a28439a39b0ac7616a18282fe9fc67fa0fa4df95dad0029f1317953"
STORAGE_SHA256 = "c50370489c4a4c1d1ae8e61770ab5005982e119fdb88f711b26ee2db5fab284d"
STUDY_ID = "estudo-principal-2026-v3-2" # @param {type:"string"}
PUBLIC_REGISTRATION_URL = "" # @param {type:"string"}
STORAGE_MODE = "auto" # @param ["auto", "local"]
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", STUDY_ID):
    raise ValueError("STUDY_ID deve conter somente letras, números, _ e -.")
ON_COLAB = False
try:
    from google.colab import drive
except ImportError:
    pass
else:
    ON_COLAB = True
LOCAL = Path(os.environ.get("COT_LOCAL_ROOT",
    "/content/cot-prospective-v3" if ON_COLAB else str(Path.cwd()/"cot-prospective-v3")))
LOCAL.mkdir(parents=True, exist_ok=True)
def frozen_checkout(ref,folder,override_name):
    override=os.environ.get(override_name)
    repo=Path(override).resolve() if override else LOCAL/folder/ref
    if not override:
        repo.mkdir(parents=True,exist_ok=True)
        if not (repo/".git").is_dir():
            subprocess.run(["git","init","--quiet",str(repo)],check=True)
            subprocess.run(["git","-C",str(repo),"remote","add","origin",
                "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git"],check=True)
        found=subprocess.run(["git","-C",str(repo),"cat-file","-e",ref+"^{commit}"],capture_output=True)
        if found.returncode:
            subprocess.run(["git","-C",str(repo),"fetch","--quiet","--depth","1","origin",ref],check=True)
        subprocess.run(["git","-C",str(repo),"checkout","--quiet","--detach",ref],check=True)
    actual=subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()
    if actual!=ref:
        raise RuntimeError("O código aberto não corresponde ao SHA do notebook.")
    return repo
REPO=frozen_checkout(SOURCE_COMMIT,"source","COT_REPO_DIR")
SETUP_REPO=frozen_checkout(SETUP_COMMIT,"setup","COT_SETUP_REPO_DIR")
PACKAGE=REPO/"experimento_cientifico_v3"
SETUP_PACKAGE=SETUP_REPO/"experimento_cientifico_v3"
support=SETUP_PACKAGE/"colab_support.py"
if hashlib.sha256(support.read_bytes()).hexdigest()!=SUPPORT_SHA256:
    raise RuntimeError("O suporte do notebook diverge do hash publicado.")
storage_support=SETUP_PACKAGE/"colab_storage.py"
if hashlib.sha256(storage_support.read_bytes()).hexdigest()!=STORAGE_SHA256:
    raise RuntimeError("O suporte de armazenamento diverge do hash publicado.")
storage_spec=importlib.util.spec_from_file_location("cot_storage_"+SETUP_COMMIT,storage_support)
storage_module=importlib.util.module_from_spec(storage_spec)
storage_spec.loader.exec_module(storage_module)
state=storage_module.prepare_storage(LOCAL,STUDY_ID,on_colab=ON_COLAB,mode=STORAGE_MODE,
    storage_override=os.environ.get("COT_STORAGE_ROOT"),
    drive_module=drive if ON_COLAB else None,
    mountpoint=os.environ.get("COT_DRIVE_MOUNTPOINT","/content/drive"))
spec=importlib.util.spec_from_file_location("cot_scientific_support_"+SETUP_COMMIT,support)
module=importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
STORAGE=state.path
STUDY=module.ScientificSession(REPO,STORAGE,SOURCE_COMMIT,SCRIPT_SHA256,ON_COLAB,
    storage_persistent=state.persistent,storage_backend=state.backend)
execute=STUDY.execute
print("Código:",SOURCE_COMMIT,"| Estudo:",STUDY_ID)
print("Suporte Colab:",SETUP_COMMIT)
print("Arquivos:",STORAGE,"| Persistência verificada:",state.persistent)


In [ ]:
# Auditoria de DEV reaproveitando os arquivos existentes (somente leitura).
import json, hashlib, importlib.util
from pathlib import Path

def verificar_dev_salvo():
    completo = True
    valores = {}
    for modo, niveis in (("sft", ("A", "C")), ("grpo", ("low", "high"))):
        resumo = STORAGE / "analysis" / modo / "dev_base_accuracy.json"
        if not resumo.exists():
            print("DEV ainda não produzido:", modo)
            completo = False
            continue
        dados = json.loads(resumo.read_text(encoding="utf-8"))
        for nivel in niveis:
            dado = dados.get(nivel)
            if dado is None:
                completo = False
                continue
            arquivo = Path(dado["file"]).resolve()
            if not arquivo.is_relative_to(STORAGE.resolve()) or not arquivo.is_file():
                raise RuntimeError("Arquivo DEV ausente ou fora do estudo: " + str(arquivo))
            conteudo = arquivo.read_bytes()
            if hashlib.sha256(conteudo).hexdigest() != dado["sha256"]:
                raise RuntimeError("SHA256 divergente no DEV; não sobrescreva o arquivo: " + str(arquivo))
            amostras = [json.loads(l) for l in conteudo.decode("utf-8").splitlines() if l.strip()]
            acertos = sum(bool(item["accuracy"]) for item in amostras)
            if (len(amostras) != dado["n"] or
                    abs(acertos / len(amostras) - dado["accuracy"]) > 1e-12):
                raise RuntimeError("Resumo DEV não corresponde aos registros: " + str(arquivo))
            valores[modo + "/" + nivel] = dado["accuracy"]
            print(f"DEV {modo.upper()} {nivel}: {acertos}/{len(amostras)} "
                  f"({dado['accuracy']:.1%}); exigido 65%")
            erros = [x for x in amostras if not x["accuracy"]]
            for item in erros[:2]:
                print("  Exemplo incorreto:", repr(item["generation"][:240]),
                      "| gabarito:", item["gold"])
    if not completo or len(valores) != 4:
        return False, False
    spec = importlib.util.spec_from_file_location("cot_dev_gate_readonly",
                                                   PACKAGE / "scientific_runtime.py")
    modulo = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modulo)
    gate = modulo.baseline_gate(STORAGE)
    if not gate["passed"]:
        print("\nTRIAGEM DEV REPROVADA:", ", ".join(gate["reasons"]))
        print("Não há treinos confirmatórios autorizados neste estudo congelado.")
    return True, gate["passed"]

DEV_COMPLETO, DEV_APROVADO = verificar_dev_salvo()
DEV_BLOQUEADO = DEV_COMPLETO and not DEV_APROVADO


In [ ]:
GPU_SESSION = None
if DEV_BLOQUEADO:
    print("DEV reprovado anteriormente. GPU não será inicializada novamente.")
else:
    GPU_SESSION=STUDY.gpu()
    if GPU_SESSION is not None:
        print("Ambiente GPU preparado.")


In [ ]:
MAX_JOBS_PER_SESSION = 1 # @param {type:"integer"}
if DEV_BLOQUEADO:
    print("Treinos GRPO não iniciados: competência DEV inferior ao mínimo do pré-registro.")
elif GPU_SESSION is not None:
    execute('prepare')
    execute('models',gpu=True)
    for modo in ("sft", "grpo"):
        if not (STORAGE/"analysis"/modo/"dev_base_accuracy.json").exists():
            execute('dev',mode=modo,gpu=True)
    DEV_COMPLETO, DEV_APROVADO = verificar_dev_salvo()
    if DEV_COMPLETO and DEV_APROVADO:
        progress = execute('train',mode='grpo',max_jobs=MAX_JOBS_PER_SESSION,gpu=True)
        print(progress['completed'],'/',progress['total'],'treinos GRPO concluídos.')
    else:
        print("Treinos GRPO pausados: triagem DEV não aprovada. Consulte as amostras acima.")
else:
    print("Treinos não iniciados: Drive/GPU indisponível.")
STUDY.export(download=False)


Com o Drive conectado, a reexecução retoma o mesmo estudo.
Se a montagem falhar, o 00 prepara uma cópia temporária para download;
os treinos aguardam armazenamento persistente. O modo local não salva no Drive.
Não altere o STUDY_ID depois de iniciar. A versão v3.2 usa um ID novo para
preservar estudos antigos. A execução confirmatória depende de pré-registro
público anterior aos treinos e de avaliação humana; essas etapas não são
substituídas por resultados de software.
